In [4]:
import re
from collections import Counter
from typing import List, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
import numpy as np

In [5]:
test_corpus = ["minecraft redstone farm", "python programming tutorial"]
vec = TfidfVectorizer()
res = vec.fit_transform(test_corpus)

print("Type of fit_transform result:", type(res))
print("Array shape:", res.toarray().shape)

Type of fit_transform result: <class 'scipy.sparse._csr.csr_matrix'>
Array shape: (2, 6)


In [6]:
def group_playlist_by_tfidf(playlist_videos: List[Dict], distance_threshold: float=0.7)->Dict[str, List[Dict]]:
    """Clusters playlist videos using TF-IDF feature extraction and agglomerative cosine distance clustering."""
    if not playlist_videos:
        return {}
    
    # Prepare raw text corpus for each video
    corpus = []
    for v in playlist_videos:
        title = v.get("title", "")
        desc = v.get("description", "")
        tags = " ".join(v.get("tags") or [])
        full_text = f"{title} {desc} {tags}".strip()
        corpus.append(full_text)
        
    if not any(corpus):
        return {"all_videos": playlist_videos}
        
    # Extract TF-IDF matrix
    vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
    tfidf_sparse = vectorizer.fit_transform(corpus)
    tfidf_matrix = np.asarray(tfidf_sparse.todense())
    feature_names = np.array(vectorizer.get_feature_names_out())
    
    # Perform agglomerative clustering with cosine distance
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=distance_threshold
    )
    labels = clustering.fit_predict(tfidf_matrix)
    
    clusters = {}
    for idx, label in enumerate(labels):
        clusters.setdefault(label, []).append((playlist_videos[idx], tfidf_matrix[idx]))
    topic_groups = {}
    for label, cluster_items in clusters.items():
        vids = [item[0] for item in cluster_items]
        vectors = np.array([item[1] for item in cluster_items])
        mean_vector = vectors.mean(axis=0)
        top_indices = mean_vector.argsort()[-2:][::-1]
        top_words = feature_names[top_indices]
        auto_label = '_'.join(top_words) if len(top_words) > 0 else f"cluster_{label}"
        topic_groups[auto_label] = vids
    return topic_groups

In [8]:
import sys
sys.path.insert(0, "../src")
from youtube_client import YouTubeClient
from topic_analysis import group_playlist_by_graph_jaccard
from candidate_retrieval import generate_topic_search_queries, filter_existing_playlist_videos
from ranking import rank_candidates

playlist_url = "https://www.youtube.com/watch?v=2xcFM9CBiOE&list=PLIdGxYqxOZEXyzTG_WXp9cujA3_xgp5Ps"
client = YouTubeClient()
print("Fetching playlist videos...")
playlist_videos = client.fetch_playlist_videos(playlist_url)
# Assuming playlist_videos is loaded from your YouTubeClient
print("=== 1. Graph-Based Jaccard Clustering ===")
graph_groups = group_playlist_by_graph_jaccard(playlist_videos, similarity_threshold=0.08)
for topic, vids in graph_groups.items():
    print(f"Topic '{topic}': {len(vids)} videos")

graph_queries = generate_topic_search_queries(graph_groups)
print("Generated Search Queries:", graph_queries)

print("\n" + "="*50 + "\n")

print("=== 2. TF-IDF + Cosine Clustering ===")
tfidf_groups = group_playlist_by_tfidf(playlist_videos, distance_threshold=0.7)
for topic, vids in tfidf_groups.items():
    print(f"Topic '{topic}': {len(vids)} videos")

tfidf_queries = generate_topic_search_queries(tfidf_groups)
print("Generated Search Queries:", tfidf_queries)

Fetching playlist videos...
=== 1. Graph-Based Jaccard Clustering ===
Topic 'com_instagram': 49 videos
Topic 'calvetti_rosario': 1 videos
Generated Search Queries: {'com_instagram': ['com instagram https'], 'calvetti_rosario': ['calvetti rosario campos']}


=== 2. TF-IDF + Cosine Clustering ===
Topic 'holy_worship': 4 videos
Topic 'mighty_jesus': 2 videos
Topic 'jesus_oh': 1 videos
Topic 'awesome_god': 1 videos
Topic 'aware_nowell': 1 videos
Topic 'grateful_just': 1 videos
Topic 'rhythm_washed': 2 videos
Topic 'forrest_frank': 4 videos
Topic 'courage_kevin': 1 videos
Topic 'come_jesus': 1 videos
Topic 'worthy_deserve': 1 videos
Topic 'tribl_moore': 2 videos
Topic 'commons_grace': 1 videos
Topic 'good_god': 2 videos
Topic 'josiah_queen': 2 videos
Topic 'alle_alleluia': 1 videos
Topic 'rush_believe': 1 videos
Topic 'god_seuworship': 1 videos
Topic 'work_god': 1 videos
Topic 'compares_kings': 1 videos
Topic 'amazing_isn': 1 videos
Topic 'time_elevationworship': 2 videos
Topic 'free_ll': 1